# Prediction Market Benchmark: LookaheadAI vs Search-R1

This notebook runs the full benchmark pipeline on a Colab A100:
1. Mount Google Drive & clone/pull repo
2. Install dependencies
3. Set API keys
4. Prepare dataset (10 balanced resolved Polymarket questions)
5. Start Exa retriever server (background)
6. Run LookaheadAI evaluation (Qwen2.5-3B-Instruct + Exa news + log-probs)
7. Run Search-R1 evaluation (SearchR1-qwen2.5-3b-grpo autonomous search)
8. Evaluate & compare results

**Runtime:** ~60-90 minutes on A100 40GB

## Step 1: Mount Drive & Set Up Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH = '/content/drive/MyDrive/Research-Lookahead-AI'
REPO_PATH  = '/content/Research-Lookahead-AI'

# Clone or pull the repo
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/YOUR_USERNAME/Research-Lookahead-AI.git {REPO_PATH}
else:
    !cd {REPO_PATH} && git pull

# Checkout the benchmark branch
!cd {REPO_PATH} && git checkout benchmark/polymarket-eval

# Add repo to Python path
import sys
sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
print(f'Working directory: {os.getcwd()}')

## Step 2: Install Dependencies

In [ ]:
%%bash
pip install -q \
    transformers>=4.40.0 \
    accelerate \
    torch \
    exa_py>=1.0.0 \
    tavily-python>=0.3.0 \
    fastapi \
    uvicorn \
    pydantic>=2.0.0 \
    datasets \
    pandas \
    pyarrow \
    requests \
    bitsandbytes

echo 'Dependencies installed.'

## Step 3: Configure API Keys

In [ ]:
import os

# ── Set your keys here ──────────────────────────────────────────────────────
os.environ['EXA_API_KEY']    = '9f57d81d-2105-40aa-9c64-b9acdbd96d61'
os.environ['RETRIEVER_URL']  = 'http://127.0.0.1:8000/retrieve'
# ────────────────────────────────────────────────────────────────────────────

# Optional: log in to HuggingFace if models are gated
# from huggingface_hub import login
# login(token='hf_...')

print('API keys configured.')

## Step 4: Prepare Benchmark Dataset

In [ ]:
# This downloads & samples 10 resolved markets from SII-WANGZJ/Polymarket_data
# Estimated time: 5-10 minutes (streams trades lazily)
!python benchmark/prepare_dataset.py

In [ ]:
import json
from pathlib import Path

markets = json.loads(Path('benchmark/data/benchmark_markets.json').read_text())
print(f'Loaded {len(markets)} benchmark markets')
print()
print('Leakage safety check:')
print('  Qwen2.5 training cutoff: ~early 2024 (released Sep 2024)')
print('  All markets below resolved >= 2025-01-01 → outcomes NOT in pretraining')
print()
for i, m in enumerate(markets):
    gt = 'YES' if m['ground_truth'] == 1 else 'NO'
    ph = f"{m['price_at_cutoff']:.3f}" if m['price_at_cutoff'] else 'N/A'
    print(f"[{i+1:2d}] [{m['category']:12s}] GT={gt} end={m['end_date']} price@cutoff={ph}")
    print(f"      {m['question'][:80]}")

## Step 5: Start Exa Retriever Server (Background)

In [ ]:
import subprocess, time, requests

# Start the FastAPI server in background
retriever_proc = subprocess.Popen(
    ['python', 'benchmark/exa_retriever_server.py'],
    env={**os.environ},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for it to come up
print('Starting Exa retriever server...', end='')
for _ in range(20):
    time.sleep(2)
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if r.status_code == 200:
            print(f' Ready! {r.json()}')
            break
    except:
        print('.', end='', flush=True)
else:
    print('\nWARNING: Retriever server may not be ready. Check logs.')

# Quick test
test_payload = {'queries': ['Will Bitcoin reach $100k?'], 'topk': 2}
resp = requests.post('http://127.0.0.1:8000/retrieve', json=test_payload)
docs = resp.json()['result'][0]
print(f'Test retrieval returned {len(docs)} documents.')
if docs:
    print(f'  First doc preview: {docs[0]["document"]["contents"][:100]}...')

## Step 6: Run LookaheadAI Evaluation

In [ ]:
# Estimated time: ~30-40 minutes for 10 markets (model load + Exa fetch + log-prob extraction)
!python benchmark/run_lookahead.py \
    --markets benchmark/data/benchmark_markets.json \
    --out     benchmark/results/lookahead_results.json \
    --exa-key {os.environ['EXA_API_KEY']} \
    --news-window-days 30

In [ ]:
la_results = json.loads(Path('benchmark/results/lookahead_results.json').read_text())
print(f'LookaheadAI: {len(la_results)} results')
correct = sum(1 for r in la_results if r['predicted_label'] == r['ground_truth'])
brier   = sum((r['predicted_prob_yes'] - r['ground_truth'])**2 for r in la_results) / len(la_results)
print(f'Accuracy: {correct}/{len(la_results)} = {correct/len(la_results)*100:.1f}%')
print(f'Brier Score: {brier:.4f}')

print('\nPer-market:')
for r in la_results:
    gt = 'YES' if r['ground_truth']==1 else 'NO'
    pred = 'YES' if r['predicted_label']==1 else 'NO'
    correct_mark = '✓' if r['predicted_label'] == r['ground_truth'] else '✗'
    print(f"  {correct_mark} [{r['category']:10}] GT={gt} pred={pred}({r['predicted_prob_yes']:.2f}) | {r['question'][:55]}")

## Step 7: Run Search-R1 Evaluation

In [ ]:
# Frees LookaheadAI model from GPU memory before loading Search-R1
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory free: {torch.cuda.memory_reserved()/1e9:.1f}GB reserved / '
      f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB total')

In [ ]:
# Estimated time: ~30-40 minutes for 10 markets
!python benchmark/run_searchr1.py \
    --markets benchmark/data/benchmark_markets.json \
    --out     benchmark/results/searchr1_results.json \
    --retriever-url http://127.0.0.1:8000/retrieve

In [ ]:
sr1_results = json.loads(Path('benchmark/results/searchr1_results.json').read_text())
print(f'Search-R1: {len(sr1_results)} results')
correct = sum(1 for r in sr1_results if r['predicted_label'] == r['ground_truth'])
brier   = sum((r['predicted_prob_yes'] - r['ground_truth'])**2 for r in sr1_results) / len(sr1_results)
print(f'Accuracy: {correct}/{len(sr1_results)} = {correct/len(sr1_results)*100:.1f}%')
print(f'Brier Score: {brier:.4f}')

print('\nPer-market:')
for r in sr1_results:
    gt = 'YES' if r['ground_truth']==1 else 'NO'
    pred = 'YES' if r['predicted_label']==1 else 'NO'
    correct_mark = '✓' if r['predicted_label'] == r['ground_truth'] else '✗'
    print(f"  {correct_mark} [{r['category']:10}] GT={gt} pred={pred}({r['predicted_prob_yes']:.2f}) "
          f"searches={r['search_count']} | {r['question'][:50]}")

## Step 8: Compare & Evaluate

In [ ]:
!python benchmark/evaluate.py \
    --lookahead benchmark/results/lookahead_results.json \
    --searchr1  benchmark/results/searchr1_results.json \
    --out       benchmark/results/comparison_report.json

In [ ]:
report = json.loads(Path('benchmark/results/comparison_report.json').read_text())

print('=== FINAL SUMMARY ===')
print(json.dumps(report['summary'], indent=2))
print()
print('=== LOOKAHEAD METRICS ===')
print(f"  Accuracy:    {report['lookahead_metrics']['accuracy']:.4f}")
print(f"  Brier Score: {report['lookahead_metrics']['brier_score']:.4f}")
print(f"  Log Loss:    {report['lookahead_metrics']['log_loss']:.4f}")
print()
print('=== SEARCH-R1 METRICS ===')
print(f"  Accuracy:    {report['searchr1_metrics']['accuracy']:.4f}")
print(f"  Brier Score: {report['searchr1_metrics']['brier_score']:.4f}")
print(f"  Log Loss:    {report['searchr1_metrics']['log_loss']:.4f}")
print(f"  Avg Searches: {report['searchr1_metrics']['avg_searches_used']}")

## Step 9: Save Results to Drive

In [ ]:
import shutil, time

timestamp = time.strftime('%Y%m%d_%H%M%S')
drive_out = f'/content/drive/MyDrive/benchmark_results_{timestamp}'
os.makedirs(drive_out, exist_ok=True)

for fname in ['lookahead_results.json', 'searchr1_results.json', 'comparison_report.json']:
    src = f'benchmark/results/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{drive_out}/{fname}')
        print(f'Saved: {drive_out}/{fname}')

# Also save the dataset used
shutil.copy('benchmark/data/benchmark_markets.json', f'{drive_out}/benchmark_markets.json')

print(f'\nAll results saved to Google Drive: {drive_out}')

## Cleanup

In [ ]:
# Stop the retriever server
retriever_proc.terminate()
print('Retriever server stopped.')